# Improve Data Quality and Class Balance
This notebook focuses on class distribution, targeted augmentation ideas for weak classes, and annotation review workflows.

## Step 1 — Check class distribution
Count labels per class from YOLO-format label files.

In [ ]:
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt

# Update this to your labels folder
labels_dir = Path("..") / "kaggle_output" / "RDD_SPLIT" / "train" / "labels"

if not labels_dir.exists():
    raise FileNotFoundError(f"labels_dir not found: {labels_dir}")

counts = Counter()
for label_file in labels_dir.glob("*.txt"):
    for line in label_file.read_text().strip().splitlines():
        if not line.strip():
            continue
        cls_id = int(line.split()[0])
        counts[cls_id] += 1

print("Class counts:", counts)

# Bar chart
if counts:
    classes = sorted(counts.keys())
    values = [counts[c] for c in classes]
    plt.figure(figsize=(8, 4))
    plt.bar([str(c) for c in classes], values)
    plt.title("Label Count per Class")
    plt.xlabel("Class ID")
    plt.ylabel("Count")
    plt.show()

## Step 2 — Targeted augmentation for weak classes
Generate a list of images that contain underrepresented classes so you can focus augmentation there.

In [ ]:
# Update this to your images folder (paired with labels)
images_dir = Path("..") / "kaggle_output" / "RDD_SPLIT" / "train" / "images"

weak_classes = {1, 4}

if not images_dir.exists():
    raise FileNotFoundError(f"images_dir not found: {images_dir}")

weak_images = []
for label_file in labels_dir.glob("*.txt"):
    txt = label_file.read_text().strip().splitlines()
    classes_in_file = {int(line.split()[0]) for line in txt if line.strip()}
    if classes_in_file & weak_classes:
        image_file = images_dir / (label_file.stem + ".jpg")
        if image_file.exists():
            weak_images.append(image_file)

print(f"Images containing weak classes: {len(weak_images)}")
print("Example images:", weak_images[:5])

## Step 2b — Visual preview of weak-class images
Quickly review a few samples to guide augmentations.

In [ ]:
import random
from PIL import Image, ImageDraw

sample_n = 50
sample_paths = random.sample(weak_images, k=min(sample_n, len(weak_images)))

ncols = 5
nrows = int((len(sample_paths) + ncols - 1) / ncols)
plt.figure(figsize=(4 * ncols, 4 * nrows))

for i, p in enumerate(sample_paths):
    img = Image.open(p).convert("RGB")
    draw = ImageDraw.Draw(img)

    label_path = labels_dir / (p.stem + ".txt")
    if label_path.exists():
        for line in label_path.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            cls_id = int(parts[0])
            if cls_id not in weak_classes:
                continue
            x, y, bw, bh = map(float, parts[1:5])
            w, h = img.size
            x1 = (x - bw / 2) * w
            y1 = (y - bh / 2) * h
            x2 = (x + bw / 2) * w
            y2 = (y + bh / 2) * h
            draw.rectangle([x1, y1, x2, y2], outline="red", width=2)
            draw.text((x1, y1), f"{cls_id}", fill="red")

    plt.subplot(nrows, ncols, i + 1)
    plt.imshow(img)
    plt.title(p.name)
    plt.axis("off")

plt.tight_layout()
plt.show()

## Step 3 — Review CLASS_0 annotations
Inspect a sample of CLASS_0 images with boxes drawn to check label consistency.

In [ ]:
from PIL import ImageDraw

class0_samples = []
for label_file in labels_dir.glob("*.txt"):
    txt = label_file.read_text().strip().splitlines()
    if any(line.strip().startswith("0 ") or line.strip().split()[0] == "0" for line in txt if line.strip()):
        image_file = images_dir / (label_file.stem + ".jpg")
        if image_file.exists():
            class0_samples.append((image_file, label_file))

print(f"CLASS_0 images: {len(class0_samples)}")

# Inspect 50-100 images
inspect_n = 60
inspect_items = random.sample(class0_samples, k=min(inspect_n, len(class0_samples)))

ncols = 3
nrows = int((len(inspect_items) + ncols - 1) / ncols)
plt.figure(figsize=(5 * ncols, 5 * nrows))

for i, (img_path, label_path) in enumerate(inspect_items):
    img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    w, h = img.size

    for line in label_path.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cls_id = int(parts[0])
        if cls_id != 0:
            continue
        x, y, bw, bh = map(float, parts[1:5])
        x1 = (x - bw / 2) * w
        y1 = (y - bh / 2) * h
        x2 = (x + bw / 2) * w
        y2 = (y + bh / 2) * h
        draw.rectangle([x1, y1, x2, y2], outline="red", width=2)

    plt.subplot(nrows, ncols, i + 1)
    plt.imshow(img)
    plt.title(img_path.name)
    plt.axis("off")

plt.tight_layout()
plt.show()